<a href="https://colab.research.google.com/github/dodi-ctrl/PhishingDetector/blob/main/DistilBERT_Phishing_Text_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Installing the libraries
!pip install -q transformers datasets torch scikit-learn pandas accelerate
print("Installation OK")

In [ ]:
# Environmental assessment
import torch
import pandas as pd
import numpy as np
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)
import warnings
warnings.filterwarnings('ignore')

# Verify GPU is available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device detected: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print("All good, we can proceed.")
else:
    print("NO GPU. Go to Runtime -> Change runtime type -> T4 GPU, then restart.")

In [ ]:
# Load the phishing email dataset from Hugging Face
print("Loading dataset")
dataset = load_dataset("zefang-liu/phishing-email-dataset")
df = pd.DataFrame(dataset['train'])

print(f"\nDataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nFirst 3 rows:")
print(df.head(3))
print(f"\nLabel distribution:")
for col in df.columns:
    if df[col].nunique() < 5:
        print(f"  {col}: {df[col].value_counts().to_dict()}")

In [ ]:
import os

# Clone repo for dataset_handling.build_eml_corpus / extract_body_text_from_eml_corpus
if not os.path.exists('PhishingDetector'):
    !git clone https://github.com/dodi-ctrl/PhishingDetector.git
import sys; sys.path.insert(0, 'PhishingDetector')

# Phishing corpora
if not os.path.exists('phishing_pot'):
    !git clone --depth 1 https://github.com/rf-peixoto/phishing_pot.git
if not os.path.exists('phishing3.mbox'):
    !wget -q https://monkey.org/~jose/phishing/phishing3.mbox

for f in ('phishing_pot', 'phishing3.mbox'):
    print(f'  {f}: {"OK" if os.path.exists(f) else "MISSING"}')

In [ ]:
# Build the .eml corpus and pull body text out for DistilBERT
from dataset_handling import build_eml_corpus, extract_body_text_from_eml_corpus

eml_df = build_eml_corpus(
    phishing_dirs=['phishing_pot/email', 'phishing_pot/emails'],
    phishing_mbox_paths=['phishing3.mbox'],
    enron_max=5000,
    nazario_after_year=2022,
)

eml_text_df = extract_body_text_from_eml_corpus(eml_df)[['text', 'label']]
print(f'EML body texts extracted: {len(eml_text_df)}')
print(f'  Phishing: {(eml_text_df.label == 1).sum()}')
print(f'  Legit:    {(eml_text_df.label == 0).sum()}')

# Note: at this point `df` from cell 3 is the raw MeAJOR DataFrame (still has
# Email Text / Email Type columns). We normalise it to {text, label} so it
# can be concatenated with eml_text_df, then let the next cell continue.
df = df.dropna(subset=['Email Text']).copy()
df['label'] = df['Email Type'].apply(lambda x: 1 if 'Phishing' in str(x) else 0)
df = df[['Email Text', 'label']].rename(columns={'Email Text': 'text'})

import pandas as pd
df = pd.concat([df, eml_text_df], ignore_index=True)
df = df[df['text'].astype(str).str.strip() != ''].reset_index(drop=True)

print(f'\nCombined corpus size: {len(df)}')
print(f'  Safe (0):     {(df.label == 0).sum()}')
print(f'  Phishing (1): {(df.label == 1).sum()}')

In [ ]:
# Preprocessing and label encoding
# Idempotent: works whether df is the raw MeAJOR DataFrame or has already been
# normalised+concatenated by the augmentation cells above.
TEXT_COL  = 'Email Text'
LABEL_COL = 'Email Type'

if TEXT_COL in df.columns and LABEL_COL in df.columns:
    df['label'] = df[LABEL_COL].apply(lambda x: 1 if 'Phishing' in str(x) else 0)
    df = df.dropna(subset=[TEXT_COL])
    df = df[df[TEXT_COL].astype(str).str.strip() != '']
    df = df[[TEXT_COL, 'label']].rename(columns={TEXT_COL: 'text'}).reset_index(drop=True)
else:
    # Already normalised (text/label columns present)
    df = df.dropna(subset=['text']).copy()
    df = df[df['text'].astype(str).str.strip() != ''].reset_index(drop=True)

print(f'Total emails after cleaning: {len(df)}')
print(f'Label distribution:')
print(f'  Safe (0):     {(df["label"] == 0).sum()}')
print(f'  Phishing (1): {(df["label"] == 1).sum()}')

text_lens = df['text'].str.len()
print(f'\nText length stats:')
print(f'  Mean: {text_lens.mean():.0f} chars | Median: {text_lens.median():.0f} | Max: {text_lens.max()}')

In [ ]:
# =====================================================
# AUGMENT TRAINING SET WITH MODERN LEGITIMATE EMAILS
# Insert this cell BEFORE the stratified train/test split
# =====================================================
# Sources:
#   - AreLit/PhishNChips    : 1000 modern workplace legit emails (synthetic but realistic)
#   - cybersectony/PhishingEmailDetectionv2.0 : ~11k legit_email samples (label=0, mix of pro emails)
#   - synthetic_legit_emails.csv : our 150 hand-templated emails (banking, university, etc.)
#
# Target after augmentation: ~12k legit modern emails added on top of current corpus.
# Expected effect: drastically reduces false positives on real bank/university/workplace emails
# while preserving phishing detection (we don't touch the phishing class).

!pip install -q datasets

from datasets import load_dataset
import pandas as pd

print("=" * 60)
print("STEP 1/4 : Loading PhishNChips (modern workplace legit emails)")
print("=" * 60)

import json

phishnchips = load_dataset("AreLit/PhishNChips", "emails")
print("PhishNChips splits:", list(phishnchips.keys()))

# Combine all splits
phishnchips_df_list = [phishnchips[s].to_pandas() for s in phishnchips.keys()]
phishnchips_full = pd.concat(phishnchips_df_list, ignore_index=True)
print(f"PhishNChips total rows: {len(phishnchips_full)}")
print(f"PhishNChips columns: {list(phishnchips_full.columns)}")
print(f"phish_label distribution:\n{phishnchips_full['phish_label'].value_counts()}")

# Filter to legitimate emails only (phish_label == 0)
phishnchips_legit_raw = phishnchips_full[phishnchips_full['phish_label'] == 0].copy()
print(f"PhishNChips legit-only rows (raw): {len(phishnchips_legit_raw)}")

# email_content is a JSON string with fields like sender, subject, body...
# We extract a flat text representation: "Subject: ...\nFrom: ...\n\n<body>"
def parse_email_json(raw):
    if pd.isna(raw):
        return ""
    if isinstance(raw, str):
        # Try parsing as JSON; if it fails, return raw as-is (already flat text).
        try:
            obj = json.loads(raw)
        except Exception:
            return raw
    elif isinstance(raw, dict):
        obj = raw
    else:
        return str(raw)

    # Walk common keys to assemble a readable email
    parts = []
    for key in ['subject', 'Subject', 'subj']:
        if key in obj and obj[key]:
            parts.append(f"Subject: {obj[key]}")
            break
    for key in ['sender', 'from', 'From']:
        if key in obj and obj[key]:
            parts.append(f"From: {obj[key]}")
            break
    for key in ['recipient', 'to', 'To']:
        if key in obj and obj[key]:
            parts.append(f"To: {obj[key]}")
            break
    body_keys = ['body', 'content', 'text', 'message', 'email_body', 'plain_text', 'html']
    body = ""
    for key in body_keys:
        if key in obj and obj[key]:
            body = str(obj[key])
            break
    if not body:
        # Fallback: dump whatever else is there
        body = " ".join(f"{k}: {v}" for k, v in obj.items() if k not in ['subject','Subject','sender','from','From','recipient','to','To'])
    parts.append("")
    parts.append(body)
    return "\n".join(parts)

phishnchips_legit_raw['text'] = phishnchips_legit_raw['email_content'].apply(parse_email_json)
phishnchips_legit = phishnchips_legit_raw[['text']].copy()
phishnchips_legit['label'] = 0
print(f"PhishNChips legit-only rows (parsed): {len(phishnchips_legit)}")
print(f"Sample parsed legit email:\n{phishnchips_legit['text'].iloc[0][:300]}\n...")


print("=" * 60)
print("STEP 2/4 : Loading cybersectony PhishingEmailDetectionv2.0")
print("=" * 60)

cyber = load_dataset("cybersectony/PhishingEmailDetectionv2.0")
print("Cybersectony splits:", list(cyber.keys()))
print("Cybersectony columns:", cyber['train'].column_names)

# Build a unified DataFrame from all splits
cyber_df_list = []
for split_name in cyber.keys():
    cyber_df_list.append(cyber[split_name].to_pandas())
cyber_full = pd.concat(cyber_df_list, ignore_index=True)
print(f"Cybersectony total rows: {len(cyber_full)}")
print(f"Label distribution:\n{cyber_full['label'].value_counts()}")

# Schema:
#   0 = legitimate_email  <-- WE WANT THIS
#   1 = phishing_email
#   2 = legitimate_url
#   3 = phishing_url
cyber_legit_email = cyber_full[cyber_full['label'] == 0][['content']].copy()
cyber_legit_email.columns = ['text']
cyber_legit_email['label'] = 0
print(f"Cybersectony legit_email rows: {len(cyber_legit_email)}")


print("=" * 60)
print("STEP 3/4 : Loading our 150 hand-templated legit emails")
print("=" * 60)

# Upload synthetic_legit_emails.csv if not already in the env
import os
if not os.path.exists('synthetic_legit_emails.csv'):
    print("synthetic_legit_emails.csv not found — please upload it.")
    from google.colab import files
    uploaded = files.upload()
    # the uploader puts files in cwd

synth_df = pd.read_csv('synthetic_legit_emails.csv')
synth_df = synth_df[['text', 'label']]
synth_df['label'] = synth_df['label'].astype(int)
print(f"Hand-templated synthetic rows: {len(synth_df)}")


print("=" * 60)
print("STEP 4/4 : Merging into the existing training corpus 'df'")
print("=" * 60)

# Drop empty / very short emails (less than 10 chars) to avoid garbage
def clean(d):
    d = d.dropna(subset=['text']).copy()
    d['text'] = d['text'].astype(str)
    d = d[d['text'].str.len() > 10].reset_index(drop=True)
    return d

phishnchips_legit = clean(phishnchips_legit)
cyber_legit_email = clean(cyber_legit_email)
synth_df = clean(synth_df)

print(f"Original df size:           {len(df)}")
print(f"PhishNChips legit to add:   {len(phishnchips_legit)}")
print(f"Cybersectony legit to add:  {len(cyber_legit_email)}")
print(f"Hand-templated to add:      {len(synth_df)}")

augmented = pd.concat([
    df,
    phishnchips_legit,
    cyber_legit_email,
    synth_df,
], ignore_index=True)

# Drop exact duplicates (some Enron emails may already be in df)
before = len(augmented)
augmented = augmented.drop_duplicates(subset=['text']).reset_index(drop=True)
after = len(augmented)
print(f"After concat:               {before}")
print(f"After dedup:                {after}  (removed {before-after} duplicates)")

# Shuffle
augmented = augmented.sample(frac=1, random_state=42).reset_index(drop=True)

# Replace df
df = augmented

print()
print("=" * 60)
print("FINAL CLASS DISTRIBUTION")
print("=" * 60)
print(df['label'].value_counts())
print()
print("Class balance ratio (legit/phishing):", round(
    df[df['label']==0].shape[0] / max(df[df['label']==1].shape[0], 1), 3
))
print()
print("✅ Augmentation complete. Continue to the train/test split cell.")

In [ ]:
# Stratified 80/20 split (keeps the same label ratio in both sets)
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df['label'],
    random_state=42
)

print(f"Training set: {len(train_df)} emails")
print(f"Test set:     {len(test_df)} emails")
print(f"\nTraining set label balance:")
print(f"  Safe:     {(train_df['label'] == 0).sum()} ({(train_df['label'] == 0).mean()*100:.1f}%)")
print(f"  Phishing: {(train_df['label'] == 1).sum()} ({(train_df['label'] == 1).mean()*100:.1f}%)")

In [ ]:
# Load DistilBERT tokenizer
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding=False,
        truncation=True,
        max_length=256,
    )

# Convert pandas DataFrames to Hugging Face Dataset format
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

# Tokenize both datasets
print("Tokenizing training set...")
train_dataset = train_dataset.map(tokenize_function, batched=True)
print("Tokenizing test set...")
test_dataset = test_dataset.map(tokenize_function, batched=True)

print(f"\nDone. Train: {len(train_dataset)} | Test: {len(test_dataset)}")
print(f"Tokenized features: {list(train_dataset.features.keys())}")

In [ ]:
# Load DistilBERT for binary classification (Safe vs Phishing)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "Safe", 1: "Phishing"},
    label2id={"Safe": 0, "Phishing": 1}
)

# Training hyperparameters
training_args = TrainingArguments(
    output_dir="./distilbert-phishing",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none",
    fp16=True,
)

# Function called after each epoch to compute evaluation metrics
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='binary'
    )
    accuracy = accuracy_score(labels, predictions)
    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

# Data collator handles padding within each batch
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Build the Trainer (new API uses processing_class instead of tokenizer)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Training setup ready.")
print(f"Model: {MODEL_NAME}")
print(f"Epochs: {training_args.num_train_epochs}")
print(f"Batch size: {training_args.per_device_train_batch_size}")
print(f"Learning rate: {training_args.learning_rate}")

In [ ]:
# Start training
print("Starting training...\n")
trainer.train()
print("\nTraining complete.")

In [ ]:
# Detailed evaluation on the test set
print("=" * 60)
print("FINAL EVALUATION ON TEST SET")
print("=" * 60)

# Get predictions on the test set
predictions = trainer.predict(test_dataset)
y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

# Detailed classification report
print("\nClassification Report:")
print(classification_report(
    y_true, y_pred,
    target_names=["Safe Email", "Phishing Email"],
    digits=4
))

# Main metrics
acc = accuracy_score(y_true, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary')

print(f"\nKey Metrics:")
print(f"  Accuracy : {acc:.4f}")
print(f"  Precision: {prec:.4f}")
print(f"  Recall   : {rec:.4f}")
print(f"  F1 Score : {f1:.4f}")

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
print(f"\nConfusion Matrix:")
print(f"                    Predicted Safe   Predicted Phishing")
print(f"  Actual Safe        {cm[0][0]:>10}        {cm[0][1]:>10}")
print(f"  Actual Phishing    {cm[1][0]:>10}        {cm[1][1]:>10}")

# Error analysis
total_safe = cm[0][0] + cm[0][1]
total_phishing = cm[1][0] + cm[1][1]
fpr = cm[0][1] / total_safe if total_safe else 0
fnr = cm[1][0] / total_phishing if total_phishing else 0

print(f"\nError Analysis:")
print(f"  False Positive Rate (Safe flagged as Phishing): {fpr:.4f} ({cm[0][1]} / {total_safe})")
print(f"  False Negative Rate (Phishing missed):           {fnr:.4f} ({cm[1][0]} / {total_phishing})")

In [ ]:
# Test the trained model on real-world email examples
def predict_email(text, model, tokenizer):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256).to(device)
    model.to(device)
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=1)[0]
    pred_class = torch.argmax(probs).item()
    label = "PHISHING" if pred_class == 1 else "SAFE"
    confidence = probs[pred_class].item()
    return label, confidence, probs

# Phishing example (typical: urgency + suspicious URL + threats)
phishing_example = """
Dear Customer,
URGENT: Your account has been temporarily suspended due to suspicious activity.
You must verify your identity within 24 hours to avoid permanent deletion.
Click here immediately: http://secure-bank-verify.tk/login?id=98234
Failure to act will result in loss of access.
Bank Security Team
"""

# Legitimate workplace email
legit_example = """
Hi team,
Just a reminder that we have our weekly standup tomorrow at 10am in Conference Room B.
The agenda is in the shared drive under "Q2 Planning".
Let me know if you have any items to add.
Thanks,
Sarah
"""

# Borderline case (file sharing - could be either)
ambiguous_example = """
Hello,
I've shared the document we discussed last week. You can access it here:
https://drive.google.com/file/d/1abc123/view
Please review and send me your feedback by Friday.
Best,
Mike
"""

print("=" * 60)
print("REAL-WORLD EMAIL TESTS")
print("=" * 60)

for name, email in [
    ("Phishing example (urgent + suspicious URL)", phishing_example),
    ("Legitimate email (workplace meeting)", legit_example),
    ("Borderline case (file sharing)", ambiguous_example),
]:
    label, conf, probs = predict_email(email, model, tokenizer)
    print(f"\n>>> {name}")
    print(f"    Prediction: {label} (confidence: {conf:.2%})")
    print(f"    Safe probability:     {probs[0].item():.4f}")
    print(f"    Phishing probability: {probs[1].item():.4f}")

In [ ]:
import shutil
import os

OUTPUT_DIR = "./distilbert_phishing_text_agent"

# Save model + tokenizer to disk
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Show what was saved
print("Files saved in", OUTPUT_DIR + ":")
for f in sorted(os.listdir(OUTPUT_DIR)):
    size_mb = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1024 / 1024
    print(f"  {f} ({size_mb:.1f} MB)")

# Create a zip archive for easy download / sharing
print("\nCreating zip archive...")
shutil.make_archive("distilbert_phishing_text_agent", 'zip', OUTPUT_DIR)
zip_size = os.path.getsize("distilbert_phishing_text_agent.zip") / 1024 / 1024
print(f"distilbert_phishing_text_agent.zip ({zip_size:.1f} MB)")

# Trigger download
from google.colab import files
files.download("distilbert_phishing_text_agent.zip")

In [ ]:
# Install LIME
!pip install -q lime
print("LIME installed")

# Upload the .eml files (a popup will open — select them from the device)
from google.colab import files
print("Select your 10 .eml files (5 spams + 5 legit)")
uploaded = files.upload()
print(f"\nUploaded {len(uploaded)} files:")
for fname in uploaded:
    print(f"  - {fname}")

In [ ]:
import email
from email import policy

def parse_eml(filename):
    """Extract subject + body text from an .eml file."""
    with open(filename, 'rb') as f:
        msg = email.message_from_binary_file(f, policy=policy.default)
    subject = msg.get('Subject', '') or ''
    sender = msg.get('From', '') or ''
    body = ''
    if msg.is_multipart():
        for part in msg.walk():
            if part.get_content_type() == 'text/plain':
                try:
                    body = part.get_content()
                    break
                except: pass
    else:
        try: body = msg.get_content()
        except: body = ''
    # If no plain text, fall back to stripping HTML
    if not body.strip():
        for part in msg.walk() if msg.is_multipart() else [msg]:
            if part.get_content_type() == 'text/html':
                try:
                    import re
                    html = part.get_content()
                    body = re.sub(r'<[^>]+>', ' ', html)
                    break
                except: pass
    return subject, sender, body.strip()

# Build list of (filename, subject, sender, body) tuples
emails_data = []
for fname in uploaded:
    if fname.endswith('.eml'):
        subj, sndr, bod = parse_eml(fname)
        emails_data.append((fname, subj, sndr, bod))

print(f"Parsed {len(emails_data)} emails:\n")
for fname, subj, sndr, bod in emails_data:
    print(f"  📧 {fname[:50]}")
    print(f"     From: {sndr[:60]}")
    print(f"     Subject: {subj[:60]}")
    print(f"     Body length: {len(bod)} chars\n")

In [ ]:
from lime.lime_text import LimeTextExplainer
import numpy as np
import torch

# LIME needs a function that takes a list of texts and returns a [N, 2] probability matrix
def predict_proba_for_lime(texts):
    """Wrap DistilBERT for LIME: input list of strings, output (N, 2) probabilities."""
    inputs = tokenizer(texts, return_tensors="pt", truncation=True,
                       max_length=256, padding=True).to(device)
    model.to(device)
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()
    return probs  # shape (N, 2): [P(safe), P(phishing)]

# Class names for LIME output
class_names = ['Safe', 'Phishing']

# Initialize the explainer
explainer = LimeTextExplainer(class_names=class_names)
print("LIME explainer ready")

In [ ]:
import matplotlib.pyplot as plt
import os

# Make a folder for LIME outputs
os.makedirs('lime_outputs', exist_ok=True)

results_summary = []

for i, (fname, subj, sndr, body) in enumerate(emails_data, 1):
    # Combine subject + body for richer context (DistilBERT only cares about text)
    full_text = f"{subj}\n\n{body}" if subj else body
    if not full_text.strip():
        print(f"[{i}] {fname[:40]} — SKIPPED (empty body)")
        continue

    # Get classification + confidence
    probs = predict_proba_for_lime([full_text])[0]
    pred_class = 'PHISHING' if probs[1] > 0.5 else 'SAFE'
    confidence = max(probs)

    # Generate LIME explanation (5000 perturbations, 10 features)
    explanation = explainer.explain_instance(
        full_text, predict_proba_for_lime,
        num_features=10, num_samples=500  # 500 = fast; bump to 5000 for higher quality
    )

    # Get top influential words for the predicted class (1 = phishing)
    top_features = explanation.as_list(label=1)  # words pushing toward phishing

    print(f"\n{'='*70}")
    print(f"[{i}] {fname}")
    print(f"  Subject: {subj[:80]}")
    print(f"  Prediction: {pred_class} (confidence {confidence:.2%})")
    print(f"  P(safe) = {probs[0]:.4f}  |  P(phishing) = {probs[1]:.4f}")
    print(f"\n  Top 10 features influencing PHISHING score:")
    for word, weight in top_features:
        arrow = "↑" if weight > 0 else "↓"
        print(f"    {arrow} {word:30} weight = {weight:+.4f}")

    # Save the matplotlib visual
    fig = explanation.as_pyplot_figure(label=1)
    fig.set_size_inches(10, 5)
    safe_fname = fname.replace('/', '_').replace('.eml', '')[:40]
    fig.savefig(f'lime_outputs/{i:02d}_{safe_fname}.png', dpi=150, bbox_inches='tight')
    plt.close(fig)

    results_summary.append({
        'file': fname,
        'subject': subj,
        'prediction': pred_class,
        'confidence': float(confidence),
        'p_safe': float(probs[0]),
        'p_phishing': float(probs[1]),
        'top_features': [(w, float(wt)) for w, wt in top_features],
    })

# Save summary as JSON
import json
with open('lime_outputs/summary.json', 'w') as f:
    json.dump(results_summary, f, indent=2)

# Zip everything for download
import shutil
shutil.make_archive('lime_outputs', 'zip', 'lime_outputs')
print(f"\n\n✓ All {len(results_summary)} LIME explanations saved")
print(f"✓ Zip ready for download")

# Trigger download
files.download('lime_outputs.zip')

In [ ]:
# =====================================================
# GRADIO INFERENCE UI WITH LIME EXPLANATIONS
# Provides a public shareable URL valid for 72h via gradio.live.
# =====================================================

!pip install -q gradio lime

import gradio as gr
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import os
from pathlib import Path
import matplotlib
matplotlib.use('Agg')   # avoid showing plots in notebook
import matplotlib.pyplot as plt
from lime.lime_text import LimeTextExplainer
from email import message_from_string
from email.policy import default as email_default_policy

# ---------- Predict helpers ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

CLASS_NAMES = ['Safe', 'Phishing']

def predict_proba_for_lime(texts):
    """Returns numpy array of shape (n, 2) with [p_safe, p_phishing] per text."""
    all_probs = []
    BATCH = 16
    for i in range(0, len(texts), BATCH):
        batch = texts[i:i+BATCH]
        enc = tokenizer(batch, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
        with torch.no_grad():
            logits = model(**enc).logits
        probs = F.softmax(logits, dim=-1).cpu().numpy()
        all_probs.append(probs)
    return np.concatenate(all_probs, axis=0)

def parse_eml_bytes(raw_bytes):
    """Parse an .eml file into (subject, body) for the model."""
    try:
        msg = message_from_string(raw_bytes.decode('utf-8', errors='ignore'), policy=email_default_policy)
    except Exception:
        msg = message_from_string(raw_bytes.decode('latin-1', errors='ignore'), policy=email_default_policy)
    subject = str(msg.get('Subject', '') or '')
    body = ""
    if msg.is_multipart():
        for part in msg.walk():
            ctype = part.get_content_type()
            if ctype == 'text/plain':
                try:
                    body = part.get_content()
                except Exception:
                    body = part.get_payload(decode=True).decode('utf-8', errors='ignore')
                break
        if not body:
            for part in msg.walk():
                if part.get_content_type() == 'text/html':
                    try:
                        body = part.get_content()
                    except Exception:
                        body = part.get_payload(decode=True).decode('utf-8', errors='ignore')
                    break
    else:
        try:
            body = msg.get_content()
        except Exception:
            payload = msg.get_payload(decode=True)
            body = payload.decode('utf-8', errors='ignore') if payload else str(msg.get_payload())
    return subject, body

# ---------- Core inference ----------
explainer = LimeTextExplainer(class_names=CLASS_NAMES)

def classify_and_explain(text, num_features=10, num_samples=500):
    if not text or not text.strip():
        return ("⚠️ Please paste an email or upload an .eml file.", None, "No input.")

    # Predict
    probs = predict_proba_for_lime([text])[0]
    p_safe, p_phish = float(probs[0]), float(probs[1])
    pred_label = CLASS_NAMES[int(np.argmax(probs))]

    # LIME explanation
    exp = explainer.explain_instance(
        text,
        predict_proba_for_lime,
        num_features=num_features,
        num_samples=num_samples,
        labels=[1],   # explain w.r.t. Phishing class
    )

    # Build matplotlib figure (better looking than fig=exp.as_pyplot_figure() default)
    feats = exp.as_list(label=1)  # list of (token, weight) sorted by absolute weight desc
    tokens = [f for f, _ in feats]
    weights = [w for _, w in feats]
    colors = ['#d9534f' if w > 0 else '#5cb85c' for w in weights]

    fig, ax = plt.subplots(figsize=(8, 4.5))
    y_pos = np.arange(len(tokens))[::-1]   # top = strongest
    ax.barh(y_pos, weights, color=colors, edgecolor='black', linewidth=0.4)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(tokens)
    ax.axvline(0, color='black', linewidth=0.5)
    ax.set_xlabel('Contribution toward "Phishing" (red) vs "Safe" (green)')
    ax.set_title(f'LIME explanation — predicted: {pred_label} ({max(p_safe, p_phish)*100:.2f}%)')
    plt.tight_layout()

    # Verdict text
    if pred_label == 'Phishing':
        verdict = (
            f"🚨  **PHISHING**  —  confidence  **{p_phish*100:.2f}%**\n\n"
            f"P(Safe) = {p_safe*100:.2f}%   |   P(Phishing) = {p_phish*100:.2f}%\n\n"
            f"Top phishing-leaning tokens (red bars), top safe-leaning tokens (green bars):"
        )
    else:
        verdict = (
            f"✅  **SAFE**  —  confidence  **{p_safe*100:.2f}%**\n\n"
            f"P(Safe) = {p_safe*100:.2f}%   |   P(Phishing) = {p_phish*100:.2f}%\n\n"
            f"The model considers this email legitimate. LIME shows the most influential tokens below."
        )

    # Build readable token detail string
    detail = "\n".join([
        f"  {token:25s}  →  {weight:+.4f}  ({'phishing' if weight > 0 else 'safe'})"
        for token, weight in feats
    ])

    return verdict, fig, detail

def classify_eml(file_obj, num_features, num_samples):
    """Wrapper for .eml upload."""
    if file_obj is None:
        return ("⚠️ No file provided.", None, "")
    with open(file_obj.name, 'rb') as f:
        raw = f.read()
    subject, body = parse_eml_bytes(raw)
    full_text = f"Subject: {subject}\n\n{body}".strip()
    verdict, fig, detail = classify_and_explain(full_text, int(num_features), int(num_samples))
    return f"**Subject parsed:** {subject}\n\n{verdict}", fig, detail


def classify_batch(files, num_features, num_samples, progress=gr.Progress()):
    """Classify a batch of .eml files and return a results table + downloadable CSV."""
    if not files:
        return None, "⚠️ No files provided. Drop one or more .eml files in the box above.", None

    rows = []
    n = len(files)
    progress(0, desc=f"Starting batch of {n} email(s)...")

    for i, file_obj in enumerate(files):
        fname = Path(file_obj.name).name
        progress((i + 0.5) / n, desc=f"[{i+1}/{n}] {fname[:60]}")

        try:
            with open(file_obj.name, 'rb') as f:
                raw = f.read()
            subject, body = parse_eml_bytes(raw)
            full_text = f"Subject: {subject}\n\n{body}".strip()

            if not full_text or len(full_text) < 5:
                rows.append({
                    "File": fname,
                    "Subject": subject[:80] if subject else "(empty)",
                    "Verdict": "ERROR",
                    "Confidence": "—",
                    "P(Safe)": "—",
                    "P(Phishing)": "—",
                    "Top phishing tokens": "(empty body)",
                    "Top safe tokens": "—",
                })
                continue

            # Classify
            probs = predict_proba_for_lime([full_text])[0]
            p_safe, p_phish = float(probs[0]), float(probs[1])
            pred_label = CLASS_NAMES[int(np.argmax(probs))]
            confidence = max(p_safe, p_phish)

            # LIME (use lower sample count by default — speed matters here)
            exp = explainer.explain_instance(
                full_text,
                predict_proba_for_lime,
                num_features=int(num_features),
                num_samples=int(num_samples),
                labels=[1],
            )
            feats = exp.as_list(label=1)
            phishing_tokens = [t for t, w in feats if w > 0][:3]
            safe_tokens = [t for t, w in feats if w < 0][:3]

            rows.append({
                "File": fname,
                "Subject": subject[:80] + ("..." if len(subject) > 80 else ""),
                "Verdict": pred_label,
                "Confidence": f"{confidence*100:.2f}%",
                "P(Safe)": f"{p_safe*100:.2f}%",
                "P(Phishing)": f"{p_phish*100:.2f}%",
                "Top phishing tokens": ", ".join(phishing_tokens) or "—",
                "Top safe tokens": ", ".join(safe_tokens) or "—",
            })
        except Exception as e:
            rows.append({
                "File": fname,
                "Subject": f"(parse error: {type(e).__name__})",
                "Verdict": "ERROR",
                "Confidence": "—",
                "P(Safe)": "—",
                "P(Phishing)": "—",
                "Top phishing tokens": str(e)[:60],
                "Top safe tokens": "—",
            })

    progress(1.0, desc="Done.")
    df = pd.DataFrame(rows)

    # Summary block
    n_total    = len(rows)
    # Verdict values are case-sensitive: "Phishing" / "Safe" / "ERROR"
    n_phishing = sum(1 for r in rows if r["Verdict"] == "Phishing")
    n_safe     = sum(1 for r in rows if r["Verdict"] == "Safe")
    n_error    = sum(1 for r in rows if r["Verdict"] == "ERROR")

    summary = (
        f"### 📊 Batch results — {n_total} email{'s' if n_total != 1 else ''} processed\n\n"
        f"- 🚨 **PHISHING**: {n_phishing}  ({n_phishing/n_total*100:.0f}%)\n"
        f"- ✅ **SAFE**:      {n_safe}  ({n_safe/n_total*100:.0f}%)\n"
        f"- ❌ **ERROR**:     {n_error}  ({n_error/n_total*100:.0f}%)\n"
    )

    # Save CSV for download
    csv_path = "/tmp/batch_results.csv"
    df.to_csv(csv_path, index=False)

    return df, summary, csv_path

# ---------- Gradio UI ----------
EXAMPLES = [
    ["""Subject: Your account has been suspended

Dear Customer,

We have detected suspicious activity on your account. Your account has been temporarily suspended for your protection.

Please verify your identity immediately by clicking the link below:
http://secure-bank-verify.com/login

If you do not verify within 24 hours, your account will be permanently closed.

Thank you,
Security Team"""],
    ["""Subject: Notification of Second Semester Course Registration

Dear Student,

This is to notify you that the Second Semester Course Registration for the 2025/2026 academic session is now open.

Kindly log into the student portal to complete your registration before the deadline.

Best regards,
Office of Academic Affairs
Nile University of Nigeria"""],
    ["""Subject: Project meeting Friday 3pm

Hi team,

Quick reminder we have the project sync Friday at 3pm in the small conference room.
Please come prepared with status updates on your work streams.

Thanks,
Sarah"""],
]

with gr.Blocks(title="Smart Phishing Detector — DistilBERT + LIME") as demo:
    gr.Markdown(
        """
        # 🛡️ Smart Phishing Detector
        ### CYB 499 Final Year Project — Nile University of Nigeria

        Paste any email text *or* upload a `.eml` file. The DistilBERT model classifies it as **Safe** or **Phishing**,
        and LIME explains *which words* drove the decision.
        """
    )

    with gr.Tabs():
        with gr.TabItem("📝 Paste email text"):
            with gr.Row():
                with gr.Column(scale=1):
                    text_in = gr.Textbox(
                        label="Email content",
                        placeholder="Paste the full email here (Subject + body)...",
                        lines=14,
                    )
                    nf = gr.Slider(5, 20, value=10, step=1, label="Number of LIME features")
                    ns = gr.Slider(200, 1500, value=500, step=100, label="LIME samples (quality vs speed)")
                    btn = gr.Button("🔍 Analyze", variant="primary")
                    gr.Examples(examples=EXAMPLES, inputs=[text_in])
                with gr.Column(scale=1):
                    verdict_out = gr.Markdown(label="Verdict")
                    fig_out = gr.Plot(label="LIME explanation")
                    detail_out = gr.Code(label="Token contributions", language="markdown")
            btn.click(fn=classify_and_explain, inputs=[text_in, nf, ns], outputs=[verdict_out, fig_out, detail_out])

        with gr.TabItem("📎 Upload .eml file"):
            with gr.Row():
                with gr.Column(scale=1):
                    file_in = gr.File(label=".eml file", file_types=['.eml'])
                    nf2 = gr.Slider(5, 20, value=10, step=1, label="Number of LIME features")
                    ns2 = gr.Slider(200, 1500, value=500, step=100, label="LIME samples")
                    btn2 = gr.Button("🔍 Analyze .eml", variant="primary")
                with gr.Column(scale=1):
                    verdict_out2 = gr.Markdown(label="Verdict")
                    fig_out2 = gr.Plot(label="LIME explanation")
                    detail_out2 = gr.Code(label="Token contributions", language="markdown")
            btn2.click(fn=classify_eml, inputs=[file_in, nf2, ns2], outputs=[verdict_out2, fig_out2, detail_out2])

        with gr.TabItem("📚 Batch .eml upload"):
            gr.Markdown(
                "Drop **multiple `.eml` files** at once. Each will be classified "
                "and explained; results appear as a sortable table and as a "
                "downloadable CSV."
            )
            with gr.Row():
                with gr.Column(scale=1):
                    files_in = gr.File(
                        label="Drop multiple .eml files",
                        file_count="multiple",
                        file_types=['.eml'],
                    )
                    nf3 = gr.Slider(5, 15, value=8, step=1,
                                    label="LIME features per email")
                    ns3 = gr.Slider(100, 800, value=250, step=50,
                                    label="LIME samples (lower = faster batches)")
                    btn3 = gr.Button("🚀 Analyze all", variant="primary")
                    summary_out = gr.Markdown()
                    csv_out = gr.File(label="📥 Download results CSV",
                                       interactive=False)
                with gr.Column(scale=2):
                    df_out = gr.Dataframe(
                        label="Per-email results",
                        wrap=True,
                        interactive=False,
                        headers=[
                            "File", "Subject", "Verdict", "Confidence",
                            "P(Safe)", "P(Phishing)",
                            "Top phishing tokens", "Top safe tokens",
                        ],
                    )
            btn3.click(
                fn=classify_batch,
                inputs=[files_in, nf3, ns3],
                outputs=[df_out, summary_out, csv_out],
            )

    gr.Markdown(
        """
        ---
        **About this demo.** The model is a fine-tuned DistilBERT trained on a curated multi-source corpus
        (Nazario, MeAJOR, phishing_pot, Enron Ham, PhishNChips, cybersectony PhishingEmailDetectionv2.0)
        plus 150 hand-templated modern legitimate emails. LIME (Ribeiro et al., 2016) is used to expose
        which tokens drove each prediction.
        """
    )

# Launch with public URL (valid 72h)
demo.launch(share=True, debug=False)

In [ ]:
# =====================================================
# QR CODE FOR THE LIVE GRADIO DEMO URL
# It reads `demo.share_url`,
# generates a high-contrast PNG QR code, displays it inline,
# and downloads it to your machine.
# =====================================================

!pip install -q "qrcode[pil]"

import qrcode
from qrcode.constants import ERROR_CORRECT_H
from IPython.display import Image, display
from google.colab import files

# ---------- Get the URL ----------
# Try to read it from the Gradio Blocks object first.
url = None
try:
    if demo.share_url:                          # auto-pickup
        url = demo.share_url
except NameError:
    pass

# Fallback: paste it manually if auto-pickup failed.
if not url:
    url = "https://PASTE-YOUR-GRADIO-URL-HERE.gradio.live"

print(f"Encoding URL: {url}")

# ---------- Build the QR ----------
qr = qrcode.QRCode(
    version=None,                                # auto-fit
    error_correction=ERROR_CORRECT_H,            # 30%, survives projector glare
    box_size=14,                                 # 14 px per module — big and crisp
    border=4,                                    # standard quiet zone
)
qr.add_data(url)
qr.make(fit=True)

img = qr.make_image(
    fill_color="#0A1F44",                        # navy — matches slide theme
    back_color="#FFFFFF",                        # white for max contrast
)

OUT = "demo_qr.png"
img.save(OUT)

print(f"✓ Saved {OUT}  ({img.size[0]}×{img.size[1]} px, {qr.modules_count}×{qr.modules_count} modules)")

# Show inline
display(Image(filename=OUT))

# Auto-download to your computer
files.download(OUT)

In [21]:
# =====================================================
# SAVE MODEL FOR LOCAL DEPLOYMENT
# Run this ONCE in Colab, after the model has been trained.
# It saves the model + tokenizer to a folder, zips them up,
# and downloads the zip to your Mac. You will then unzip into
# your local project folder and run app.py from PyCharm.
# =====================================================

import os
import shutil

# ---------- 1. Save model + tokenizer to a folder ----------
SAVE_DIR = "phishing_model"
if os.path.isdir(SAVE_DIR):
    shutil.rmtree(SAVE_DIR)
os.makedirs(SAVE_DIR, exist_ok=True)

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print(f"✓ Model + tokenizer saved to {SAVE_DIR}/")
print("Contents:")
for f in sorted(os.listdir(SAVE_DIR)):
    size_mb = os.path.getsize(os.path.join(SAVE_DIR, f)) / 1024 / 1024
    print(f"   {f:30s}  {size_mb:7.2f} MB")

# ---------- 2. Zip it up ----------
ZIP_PATH = "phishing_model.zip"
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)
shutil.make_archive("phishing_model", "zip", SAVE_DIR)

zip_size_mb = os.path.getsize(ZIP_PATH) / 1024 / 1024
print(f"\n✓ Zipped to {ZIP_PATH}  ({zip_size_mb:.1f} MB)")

# ---------- 3. Download to your Mac ----------
from google.colab import files
files.download(ZIP_PATH)
print("\n📥 Download starting. Save it somewhere you'll remember (e.g. ~/Downloads/).")
print("Then unzip it INTO a folder called 'model/' inside your local project.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✓ Model + tokenizer saved to phishing_model/
Contents:
   config.json                        0.00 MB
   model.safetensors                255.43 MB
   tokenizer.json                     0.68 MB
   tokenizer_config.json              0.00 MB

✓ Zipped to phishing_model.zip  (235.7 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


📥 Download starting. Save it somewhere you'll remember (e.g. ~/Downloads/).
Then unzip it INTO a folder called 'model/' inside your local project.
